In [1]:

#필요한 라이브러리 없는 경우 다운 !pip install pandas numpy requests scipy matplotlib seaborn folium tqdm plotly
# 라이브러리
import pandas as pd
import numpy as np
import re
import requests
from scipy import stats
from datetime import datetime
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import scipy.stats as stats
from scipy.optimize import curve_fit
import shutil
from matplotlib import font_manager, rc
import subprocess
import sys
import os
import random
import glob
import time 
import folium
from folium.plugins import HeatMap
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm



In [2]:
# 데이터 프레임 읽는 부분
import pandas as pd
#(csv파일 다운받은 경로 입력, 반드시 수정)
file_path = "/home/addd/Downloads/서울시 상권분석서비스(소득소비-행정동).csv"

# CSV 파일을 데이터프레임으로 읽어오기
df = pd.read_csv(file_path, encoding='cp949')

# 데이터프레임 출력
df

,기준_년분기_코드,행정동_코드,행정동_코드_명,월_평균_소득_금액,소득_구간_코드,지출_총금액,식료품_지출_총금액,의류_신발_지출_총금액,생활용품_지출_총금액,의료비_지출_총금액,교통_지출_총금액,교육_지출_총금액,유흥_지출_총금액,여가_문화_지출_총금액,기타_지출_총금액,음식_지출_총금액
0,20231,11680660,개포1동,5336373,9,250506000,70320000,8632000,1380000,134557000,0,18870000,0,0,5140000,11607000
1,20231,11110550,부암동,3647449,7,1385759000,232200000,39726000,12611000,101703000,109837000,78943000,2652000,131748000,6197000,670142000
2,20231,11110640,이화동,3024996,7,10622974000,1035923000,213685000,41488000,2776380000,30454000,2965305000,176154000,207416000,181317000,2994852000
3,20231,11110515,청운효자동,3780222,8,2784933000,880544000,197642000,57935000,121901000,25568000,53711000,86449000,171279000,67694000,1122210000
4,20231,11140590,광희동,3229604,7,11668791000,1282600000,4164653000,104873000,733983000,137109000,21347000,191172000,2261477000,407760000,2363817000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8070,20224,11710641,문정1동,3568357,7,4066688000,1037337000,159114000,18946000,743394000,111789000,110344000,119378000,653474000,121129000,991783000
8071,20224,11710646,장지동,3263464,7,3726339000,1310415000,36856000,15776000,663267000,334925000,331217000,85082000,195835000,80583000,672383000
8072,20224,11710647,위례동,4202887,8,4953932000,929585000,63524000,7286000,543822000,1228796000,662566000,29008000,196900000,195254000,1097191000
8073,20224,11740600,천호1동,2473111,6,2965499000,766647000,109813000,71949000,475302000,56654000,218323000,48717000,411595000,142343000,664156000


In [3]:
import pandas as pd
# 법정동 코드 정보 다운로드(csv 다운 받은 경로 입력)
file_path = "/home/addd/Downloads/INTERN_YUSEONGMIN/EDA/df_hangjungdong_code_data_2024.csv"

# CSV 파일을 데이터프레임으로 읽어오기

df_1 = pd.read_csv(file_path)

# 시군구코드+(0이 아닌)행정동코드
df_1['시군구_행정동'] = df_1.apply(lambda row: str(row['시군구코드']) + str(row['행정동코드']) if row['행정동코드'] != 0 else '', axis=1)

# 데이터프레임 출력
df_1

,시군구코드,법정동코드,행정동코드,시도명,시군구명,법정동명,행정동명,적용시작일,적용만료일,lat,lng,시군구_행정동
0,11680,10300,675,서울특별시,강남구,개포동,개포3동,20221223,99991231,37.596274,127.141984,11680675
1,11680,10600,675,서울특별시,강남구,대치동,개포3동,20221223,99991231,37.596274,127.141984,11680675
2,11680,11400,675,서울특별시,강남구,일원동,개포3동,20221223,99991231,37.596274,127.141984,11680675
3,11740,10300,525,서울특별시,강동구,상일동,상일제1동,20210701,99991231,37.596274,127.141984,11740525
4,11740,10300,526,서울특별시,강동구,상일동,상일제2동,20210701,99991231,37.596274,127.141984,11740526
...,...,...,...,...,...,...,...,...,...,...,...,...
2350,11110,11100,520,서울특별시,종로구,옥인동,효자동,19880423,20081031,37.583070,126.970580,11110520
2351,11230,10500,600,서울특별시,동대문구,답십리동,답십리제1동,19880423,99991231,37.596274,127.141984,11230600
2352,11110,11600,590,서울특별시,종로구,도렴동,세종로동,19880423,19981201,37.596274,127.141984,11110590
2353,11110,11700,590,서울특별시,종로구,당주동,세종로동,19880423,19981201,37.596274,127.141984,11110590


In [4]:
# 시군구 코드+행정동코드가 같은 것들만 남기고 싹 지우기
df_1 = df_1.drop_duplicates(subset='시군구_행정동')

df_1.loc[df_1['행정동명'] == '청운효자동']


,시군구코드,법정동코드,행정동코드,시도명,시군구명,법정동명,행정동명,적용시작일,적용만료일,lat,lng,시군구_행정동
44,11110,10800,515,서울특별시,종로구,통인동,청운효자동,20081101,99991231,37.584083,126.970636,11110515


In [5]:
# '행정동코드' 컬럼의 데이터 타입을 문자열로 변환
df['행정동_코드'] = df['행정동_코드'].astype(str)

# 병합 시도
merged_df = pd.merge(df, df_1, left_on=['행정동_코드'], right_on=['시군구_행정동'], how='inner')

# df_1과 행 개수가 같음을 확인 가능
merged_df


,기준_년분기_코드,행정동_코드,행정동_코드_명,월_평균_소득_금액,소득_구간_코드,지출_총금액,식료품_지출_총금액,의류_신발_지출_총금액,생활용품_지출_총금액,의료비_지출_총금액,...,행정동코드,시도명,시군구명,법정동명,행정동명,적용시작일,적용만료일,lat,lng,시군구_행정동
0,20231,11680660,개포1동,5336373,9,250506000,70320000,8632000,1380000,134557000,...,660,서울특별시,강남구,포이동,개포제1동,19880423,19880630,37.596274,127.141984,11680660
1,20232,11680660,개포1동,5336373,9,270218000,76475000,16223000,787000,142538000,...,660,서울특별시,강남구,포이동,개포제1동,19880423,19880630,37.596274,127.141984,11680660
2,20233,11680660,개포1동,5336373,9,292174000,84510000,11300000,2046000,160125000,...,660,서울특별시,강남구,포이동,개포제1동,19880423,19880630,37.596274,127.141984,11680660
3,20191,11680660,개포1동,4197165,8,189850000,72649000,7548000,1557000,8987000,...,660,서울특별시,강남구,포이동,개포제1동,19880423,19880630,37.596274,127.141984,11680660
4,20204,11680660,개포1동,5336373,9,152044000,80731000,10833000,2741000,28371000,...,660,서울특별시,강남구,포이동,개포제1동,19880423,19880630,37.596274,127.141984,11680660
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8070,20214,11740660,성내3동,2897346,6,6471776000,1672929000,181590000,120006000,861826000,...,660,서울특별시,강동구,성내동,성내제3동,19880423,99991231,37.596274,127.141984,11740660
8071,20221,11740660,성내3동,2897346,6,6099546000,1722666000,161685000,94348000,764340000,...,660,서울특별시,강동구,성내동,성내제3동,19880423,99991231,37.596274,127.141984,11740660
8072,20222,11740660,성내3동,2897346,6,7116476000,1705942000,196414000,122648000,920183000,...,660,서울특별시,강동구,성내동,성내제3동,19880423,99991231,37.596274,127.141984,11740660
8073,20223,11740660,성내3동,2897346,6,7035114000,1766281000,144648000,126212000,867046000,...,660,서울특별시,강동구,성내동,성내제3동,19880423,99991231,37.596274,127.141984,11740660


In [6]:
column_names = merged_df.columns.tolist()
print(column_names)
merged_df['행정동_전체_명'] = merged_df['시도명'] + ' ' + merged_df['시군구명'] + ' ' + merged_df['행정동_코드_명']


['기준_년분기_코드', '행정동_코드', '행정동_코드_명', '월_평균_소득_금액', '소득_구간_코드', '지출_총금액', '식료품_지출_총금액', '의류_신발_지출_총금액', '생활용품_지출_총금액', '의료비_지출_총금액', '교통_지출_총금액', '교육_지출_총금액', '유흥_지출_총금액', '여가_문화_지출_총금액', '기타_지출_총금액', '음식_지출_총금액', '시군구코드', '법정동코드', '행정동코드', '시도명', '시군구명', '법정동명', '행정동명', '적용시작일', '적용만료일', 'lat', 'lng', '시군구_행정동']


In [13]:
import os, json
import pandas as pd
import plotly.express as px

with open('/home/addd/Downloads/hangjeongdong_서울특별시.geojson', 'r') as f:
    seoul_geo = json.load(f)

    
mapbox_style = "carto-positron"
center = {"lat": 37.5665, "lon": 126.9780}
zoom = 10
opacity = 0.5

# 서울 행정동 경계 그리기
fig = px.choropleth_mapbox(merged_df, 
                           geojson=seoul_geo, 
                           locations='행정동_전체_명', 
                           color='월_평균_소득_금액',
                           featureidkey='properties.adm_nm',
                           center={"lat": 37.563383, "lon": 126.996039},
                           mapbox_style='carto-positron',
                           zoom=9.5,
                           # 소득 많은 순서대로 빨, 노, 초, 파
                           color_continuous_scale=[(0, 'black'), (0.01, 'blue'), (0.3, '#87CEEB'),(0.5, '#90EE90'),(0.7, 'yellow'), (1, 'red')],
                           opacity=0.5,
                           title='월 평균 소득 금액에 따른 서울시 행정동별 코로플레스 맵'
                          )


fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

# HTML 파일로 저장
fig.write_html("income_choropleth_map.html")

In [36]:
print(seoul_geo['features'][0]['properties'])


{'OBJECTID': 1, 'adm_nm': '서울특별시 종로구 사직동', 'adm_cd': '1101053', 'adm_cd2': '1111053000', 'sgg': '11110', 'sido': '11', 'sidonm': '서울특별시', 'sggnm': '종로구'}
